# Diagnóstico de calidad de datos – Siniestralidad vial en Colombia
**GA1-220501094-01-AA1-EV05** · Preparación, exploración y documentación

| | |
|---|---|
| **Proyecto** | Análisis de la siniestralidad vial en Colombia |
| **Pregunta analítica** | ¿Qué departamentos, municipios y sectores viales concentran la mayor cantidad de lesionados y fallecidos por accidentes de tránsito en Colombia, y cómo ha evolucionado esa siniestralidad en el tiempo? |
| **Herramientas** | Python 3 · Pandas |

> Este cuaderno **diagnostica** la calidad de los datos. No ejecuta la limpieza completa: solo identifica, evidencia y documenta problemas, y propone acciones.

## Momento 1. Reconocimiento del proyecto y de los datasets

| # | Archivo | Fuente | Contenido | Relación con la pregunta |
|---|---|---|---|---|
| 1 | `lesiones_accidentes_transito.csv` | Datos Abiertos Colombia (Policía Nacional) | Registros de personas lesionadas en accidentes de tránsito por fecha, departamento y municipio, con la columna `CANTIDAD` de lesionados. | Permite medir **cuántos lesionados** hay, **dónde** (departamento/municipio) y **cuándo** (serie temporal). |
| 2 | `sectores_criticos_siniestralidad_vial.csv` | Datos Abiertos Colombia (sector transporte / ANI – INVÍAS) | Puntos críticos de la red vial nacional identificados con análisis de puntos calientes (Getis-Ord Gi*): z-score, p-value, fallecidos, tramo, coordenadas y municipio. | Permite identificar **qué sectores viales** concentran fallecidos y cruzarlos con los municipios del dataset 1. |

**Aspectos de calidad a revisar:** completitud (valores faltantes y marcadores como `<Null>`, `sd`, `SIN ESTABLECER`), tipos de datos (fechas, códigos DIVIPOLA, decimales con coma), duplicados, rangos válidos (coordenadas en Colombia, p-values entre 0 y 1), consistencia de nombres entre datasets (tildes) y cobertura temporal.

## Momento 2. Preparación del entorno y carga de datos

In [1]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

# Ruta a los datos (el notebook se ejecuta desde la carpeta notebooks/)
DATA_RAW = Path("..") / "data" / "raw"
if not DATA_RAW.exists():          # por si se ejecuta desde la raíz del repositorio
    DATA_RAW = Path("data") / "raw"
list(DATA_RAW.glob("*.csv"))

Python: 3.13.18
Pandas: 3.0.5


[PosixPath('../data/raw/lesiones_accidentes_transito.csv'),
 PosixPath('../data/raw/sectores_criticos_siniestralidad_vial.csv')]

Se cargan ambos archivos con `pd.read_csv()` **sin parámetros adicionales**, tal como llegan de la fuente. Así el diagnóstico refleja cómo Pandas interpreta los datos originales (los problemas de tipos aparecen precisamente por esto). Cada dataset queda en un DataFrame independiente:
- `df_lesiones` → lesionados en accidentes de tránsito
- `df_sectores` → sectores críticos de siniestralidad vial

In [2]:
df_lesiones = pd.read_csv(DATA_RAW / "lesiones_accidentes_transito.csv", encoding="utf-8")
df_sectores = pd.read_csv(DATA_RAW / "sectores_criticos_siniestralidad_vial.csv", encoding="utf-8")

print("df_lesiones cargado:", df_lesiones.shape)
print("df_sectores cargado:", df_sectores.shape)

df_lesiones cargado: (644042, 6)
df_sectores cargado: (316, 13)


## Momento 3. Inspección estructural · Dataset 1 – Lesiones en accidentes de tránsito

### head() – primeros registros

In [3]:
df_lesiones.head()

,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,CANTIDAD
0,30/09/2026,76,VALLE DEL CAUCA,76001,CALI,1
1,30/09/2026,76,VALLE DEL CAUCA,76001,CALI,1
2,30/09/2026,76,VALLE DEL CAUCA,76001,CALI,1
3,30/09/2026,76,VALLE DEL CAUCA,76001,CALI,1
4,30/09/2026,70,SUCRE,70001,SINCELEJO,1


### tail() – últimos registros

In [4]:
df_lesiones.tail()

,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,CANTIDAD
644037,01/01/2003,52,NARIÑO,52838,TUQUERRES,1
644038,01/01/2003,68,SANTANDER,68276,FLORIDABLANCA,1
644039,01/01/2003,68,SANTANDER,68307,GIRON,1
644040,01/01/2003,23,CORDOBA,23350,LA APARTADA,1
644041,01/01/2003,41,HUILA,41396,LA PLATA,1


### shape – filas y columnas

In [5]:
print("Filas:", df_lesiones.shape[0], "| Columnas:", df_lesiones.shape[1])

Filas: 644042 | Columnas: 6


### columns – nombres de variables

In [6]:
df_lesiones.columns

Index(['FECHA HECHO', 'COD_DEPTO', 'DEPARTAMENTO', 'COD_MUNI', 'MUNICIPIO', 'CANTIDAD'], dtype='str')

### dtypes – tipos de datos

In [7]:
df_lesiones.dtypes

FECHA HECHO       str
COD_DEPTO       int64
DEPARTAMENTO      str
COD_MUNI        int64
MUNICIPIO         str
CANTIDAD        int64
dtype: object

### info() – estructura y no nulos

In [8]:
df_lesiones.info()

<class 'pandas.DataFrame'>
RangeIndex: 644042 entries, 0 to 644041
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   FECHA HECHO   644042 non-null  str  
 1   COD_DEPTO     644042 non-null  int64
 2   DEPARTAMENTO  644042 non-null  str  
 3   COD_MUNI      644042 non-null  int64
 4   MUNICIPIO     644042 non-null  str  
 5   CANTIDAD      644042 non-null  int64
dtypes: int64(3), str(3)
memory usage: 29.5 MB


### describe() – estadísticas de variables numéricas

In [9]:
df_lesiones.describe()

,COD_DEPTO,COD_MUNI,CANTIDAD
count,644042.000000,644042.000000,644042.000000
mean,40.912543,41116.832146,1.375421
std,27.441612,27449.577990,1.626253
min,0.000000,0.000000,1.000000
25%,13.000000,13442.000000,1.000000
50%,41.000000,41298.000000,1.000000
75%,68.000000,68307.000000,1.000000
max,99.000000,99773.000000,91.000000


### describe(include='object') – variables de texto

In [10]:
df_lesiones.describe(include='object')

/tmp/ipykernel_275/1821847124.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_lesiones.describe(include='object')


,FECHA HECHO,DEPARTAMENTO,MUNICIPIO
count,644042,644042,644042
unique,8674,34,1003
top,20/09/2026,VALLE DEL CAUCA,BOGOTA D.C.
freq,254,84498,64044


## Momento 3. Inspección estructural · Dataset 2 – Sectores críticos de siniestralidad vial

### head() – primeros registros

In [11]:
df_sectores.head()

,ID_MT,ENTIDAD,GiZScore,Fallecidos,GiPValue,Tramo,Nombre,Latitud,Longitud,PR,Municipio,Departamento,divipola
0,4G013,ANI,"1,771262",11,"0,076517",Bogotá - Villavicencio,IP – Chirajara - Fundadores,"4,452987","-74,046677",10,CHIPAQUE,CUNDINAMARCA,25178
1,4G013,ANI,"3,818008",23,"0,000135",Bogotá - Villavicencio,IP – Chirajara - Fundadores,"4,228251","-73,818342",58,GUAYABETAL,CUNDINAMARCA,25335
2,4G013,ANI,"2,796662",1,"0,005163",Bogotá - Villavicencio,IP – Chirajara - Fundadores,"4,203601","-73,807188",60,GUAYABETAL,CUNDINAMARCA,25335
3,4G013,ANI,"1,725132",1,"0,084504",Bogotá - Villavicencio,IP – Chirajara - Fundadores,"4,121453","-73,643971",85,VILLAVICENCIO,META,50001
4,<Null>,INVIAS-OTROS,"1,725132",13,"0,084504",Granada - Villavicencio,NaN,"4,115757","-73,647497",sd,VILLAVICENCIO,META,50001


### tail() – últimos registros

In [12]:
df_sectores.tail()

,ID_MT,ENTIDAD,GiZScore,Fallecidos,GiPValue,Tramo,Nombre,Latitud,Longitud,PR,Municipio,Departamento,divipola
311,3G012,ANI,"2,699465",12,"0,006945",Bosconia - Cienaga,Ruta del Sol 3,"9,780252","-73,748405",62,EL PASO,CESAR,20250
312,4G026,ANI,"6,546778",25,0,Chigorodó-Turbo,Autopistas al Mar 2,"8,005361","-76,640697",16,TURBO,ANTIOQUIA,5837
313,3G012,ANI,"2,403517",11,"0,016238",Bosconia - Cienaga,Ruta del Sol 3,"10,31016","-74,046928",8,ALGARROBO,MAGDALENA,47030
314,4G016,ANI,"2,699465",12,"0,006945",San Pelayo-Lorica,IP – Antioquia – Bolívar,"9,030229","-75,827469",25,COTORRA,CÓRDOBA,23300
315,4G017,ANI,"4,771095",19,"0,000002",Neiva-Girardot,IP – Neiva – Espinal - Girardot,"3,728237","-75,078416",97,COYAIMA,TOLIMA,73217


### shape – filas y columnas

In [13]:
print("Filas:", df_sectores.shape[0], "| Columnas:", df_sectores.shape[1])

Filas: 316 | Columnas: 13


### columns – nombres de variables

In [14]:
df_sectores.columns

Index(['ID_MT', 'ENTIDAD', 'GiZScore', 'Fallecidos', 'GiPValue', 'Tramo', 'Nombre', 'Latitud', 'Longitud', 'PR', 'Municipio', 'Departamento', 'divipola'], dtype='str')

### dtypes – tipos de datos

In [15]:
df_sectores.dtypes

ID_MT             str
ENTIDAD           str
GiZScore          str
Fallecidos      int64
GiPValue          str
Tramo             str
Nombre            str
Latitud           str
Longitud          str
PR                str
Municipio         str
Departamento      str
divipola        int64
dtype: object

### info() – estructura y no nulos

In [16]:
df_sectores.info()

<class 'pandas.DataFrame'>
RangeIndex: 316 entries, 0 to 315
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ID_MT         316 non-null    str  
 1   ENTIDAD       316 non-null    str  
 2   GiZScore      316 non-null    str  
 3   Fallecidos    316 non-null    int64
 4   GiPValue      316 non-null    str  
 5   Tramo         293 non-null    str  
 6   Nombre        141 non-null    str  
 7   Latitud       316 non-null    str  
 8   Longitud      316 non-null    str  
 9   PR            316 non-null    str  
 10  Municipio     316 non-null    str  
 11  Departamento  316 non-null    str  
 12  divipola      316 non-null    int64
dtypes: int64(2), str(11)
memory usage: 32.2 KB


### describe() – estadísticas de variables numéricas

In [17]:
df_sectores.describe()

,Fallecidos,divipola
count,316.000000,316.000000
mean,11.329114,41875.246835
std,7.072363,26052.206280
min,1.000000,5045.000000
25%,7.000000,20060.000000
50%,10.000000,41770.000000
75%,14.000000,68991.500000
max,52.000000,81794.000000


### describe(include='object') – variables de texto

In [18]:
df_sectores.describe(include='object')

/tmp/ipykernel_275/3024645946.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_sectores.describe(include='object')


,ID_MT,ENTIDAD,GiZScore,GiPValue,Tramo,Nombre,Latitud,Longitud,PR,Municipio,Departamento
count,316,316,316,316,293,141,316,316,316,316,316
unique,25,4,102,79,85,24,312,312,105,132,22
top,<Null>,ANI,"2,403517",0,El Hatillo - Girardota - Copacabana - Bello - ...,IP – Ampliación a Tercer Carril Doble Calzad...,"9,030229","-75,827469",sd,BELLO,VALLE DEL CAUCA
freq,175,141,18,34,15,14,2,2,42,14,60


### Interpretación de la inspección estructural

**df_lesiones** – 644.042 filas × 6 columnas. Cada fila es un registro de lesionados en una fecha y municipio; `CANTIDAD` indica cuántas personas. Pandas leyó `FECHA HECHO` como texto (no como fecha) y convirtió `COD_DEPTO` y `COD_MUNI` a enteros, lo que **elimina los ceros iniciales** de los códigos DIVIPOLA (`08` → `8`, `05001` → `5001`). `CANTIDAD` va de 1 a 91, con mediana 1: la mayoría de registros son de una persona.

**df_sectores** – 316 filas × 13 columnas. Cada fila es un punto crítico. Solo `Fallecidos` y `divipola` quedaron numéricas; `GiZScore`, `GiPValue`, `Latitud` y `Longitud` quedaron como **texto** porque usan **coma decimal** (`4,452987`). `PR` mezcla números con el texto `sd`. `Tramo` y `Nombre` tienen nulos.

## Tabla de diagnóstico estructural (generada con Pandas)

In [19]:
def resumen_tipos(df):
    return ", ".join(f"{t}: {n}" for t, n in df.dtypes.astype(str).value_counts().items())

tabla_estructural = pd.DataFrame([
    {"Dataset": "df_lesiones", "Filas": df_lesiones.shape[0], "Columnas": df_lesiones.shape[1],
     "Tipos de datos": resumen_tipos(df_lesiones),
     "Observaciones": "Fecha como texto; códigos DIVIPOLA leídos como int (pierden cero inicial); sin nulos explícitos."},
    {"Dataset": "df_sectores", "Filas": df_sectores.shape[0], "Columnas": df_sectores.shape[1],
     "Tipos de datos": resumen_tipos(df_sectores),
     "Observaciones": "Decimales con coma -> métricas y coordenadas como texto; nulos en Tramo y Nombre; marcadores '<Null>' y 'sd'."},
])
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,df_lesiones,644042,6,"str: 3, int64: 3",Fecha como texto; códigos DIVIPOLA leídos como...
1,df_sectores,316,13,"str: 11, int64: 2",Decimales con coma -> métricas y coordenadas c...


## Momento 4. Diagnóstico inicial de calidad

### 4.1 Valores faltantes (nulos explícitos)

In [20]:
def faltantes(df):
    n = df.isna().sum()
    return pd.DataFrame({"nulos": n, "%": (n / len(df) * 100).round(2)}).query("nulos > 0")

print("df_lesiones:"); display(faltantes(df_lesiones) if faltantes(df_lesiones).size else "Sin nulos explícitos")
print("df_sectores:"); display(faltantes(df_sectores))

df_lesiones:


'Sin nulos explícitos'

df_sectores:


,nulos,%
Tramo,23,7.28
Nombre,175,55.38


### 4.2 Valores faltantes ocultos (marcadores de texto)
Algunos vacíos no aparecen como `NaN` sino como textos: `<Null>`, `sd` (sin dato) o `SIN ESTABLECER`.

In [21]:
marcadores = ["<Null>", "sd", "SD", "SIN ESTABLECER", "NO REPORTA", "N/A", ""]

def faltantes_ocultos(df):
    filas = []
    for col in df.select_dtypes(include=["object", "string"]).columns:
        s = df[col].astype("string").str.strip()
        for m in marcadores:
            n = (s == m).sum()
            if n:
                filas.append({"columna": col, "marcador": m, "registros": int(n), "%": round(n / len(df) * 100, 2)})
    return pd.DataFrame(filas)

print("df_lesiones:"); display(faltantes_ocultos(df_lesiones))
print("df_sectores:"); display(faltantes_ocultos(df_sectores))

df_lesiones:


,columna,marcador,registros,%
0,DEPARTAMENTO,SIN ESTABLECER,3,0.0
1,MUNICIPIO,SIN ESTABLECER,11,0.0


df_sectores:


,columna,marcador,registros,%
0,ID_MT,<Null>,175,55.38
1,PR,sd,42,13.29


In [22]:
# Registros de lesiones con ubicación no establecida
df_lesiones[(df_lesiones["DEPARTAMENTO"] == "SIN ESTABLECER") | (df_lesiones["MUNICIPIO"] == "SIN ESTABLECER")]

,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,CANTIDAD
131762,24/12/2023,52,NARIÑO,52000,SIN ESTABLECER,1
186664,21/11/2022,52,NARIÑO,52000,SIN ESTABLECER,1
410916,30/09/2016,0,SIN ESTABLECER,0,SIN ESTABLECER,1
438274,01/08/2015,0,SIN ESTABLECER,0,SIN ESTABLECER,2
443980,25/03/2015,0,SIN ESTABLECER,0,SIN ESTABLECER,2
573825,29/03/2008,15,BOYACA,15000,SIN ESTABLECER,1
601762,19/06/2006,15,BOYACA,15000,SIN ESTABLECER,1
608427,24/12/2005,88,SAN ANDRES ISLAS,88000,SIN ESTABLECER,1
614017,23/05/2005,25,CUNDINAMARCA,25000,SIN ESTABLECER,1
617331,30/01/2005,25,CUNDINAMARCA,25000,SIN ESTABLECER,1


### 4.3 Tipos de datos

**Fecha:** `FECHA HECHO` es texto. Se verifica si todas las fechas son convertibles con formato día/mes/año y su rango.

In [23]:
fecha = pd.to_datetime(df_lesiones["FECHA HECHO"], format="%d/%m/%Y", errors="coerce")
print("Fechas no convertibles:", fecha.isna().sum())
print("Rango:", fecha.min().date(), "a", fecha.max().date())

Fechas no convertibles: 0
Rango: 2003-01-01 a 2026-09-30


**Códigos DIVIPOLA:** deben ser texto de 2 (departamento) y 5 (municipio) dígitos. Al leerse como enteros se pierden los ceros a la izquierda.

In [24]:
print("COD_DEPTO con 1 dígito:", (df_lesiones["COD_DEPTO"].astype(str).str.len() == 1).sum())
print("COD_MUNI con < 5 dígitos:", (df_lesiones["COD_MUNI"].astype(str).str.len() < 5).sum())
print("divipola (sectores) con < 5 dígitos:", (df_sectores["divipola"].astype(str).str.len() < 5).sum())
df_lesiones[df_lesiones["DEPARTAMENTO"] == "ANTIOQUIA"][["COD_DEPTO", "COD_MUNI", "MUNICIPIO"]].drop_duplicates().head()

COD_DEPTO con 1 dígito: 80832


COD_MUNI con < 5 dígitos: 80832
divipola (sectores) con < 5 dígitos: 43


,COD_DEPTO,COD_MUNI,MUNICIPIO
85,5,5045,APARTADO
88,5,5088,BELLO
120,5,5001,MEDELLIN
143,5,5266,ENVIGADO
338,5,5440,MARINILLA


**Decimales con coma** en `df_sectores`: se comprueba que, al reemplazar la coma por punto, los valores son numéricos válidos.

In [25]:
cols_coma = ["GiZScore", "GiPValue", "Latitud", "Longitud"]
prueba = df_sectores[cols_coma].apply(lambda c: pd.to_numeric(c.str.replace(",", ".", regex=False), errors="coerce"))
print("Valores no convertibles por columna:"); print(prueba.isna().sum())
prueba.describe()

Valores no convertibles por columna:
GiZScore    0
GiPValue    0
Latitud     0
Longitud    0
dtype: int64


,GiZScore,GiPValue,Latitud,Longitud
count,316.000000,316.000000,316.000000,316.000000
mean,3.192678,0.021339,5.674207,-75.172643
std,1.635115,0.026702,2.748538,1.438916
min,1.659538,0.000000,0.821057,-78.787178
25%,2.107570,0.000103,3.678333,-76.320459
50%,2.699465,0.006945,4.749676,-75.430933
75%,3.883253,0.035068,7.804252,-73.952275
max,14.537352,0.097007,11.519362,-71.724108


**Punto de referencia `PR`:** mezcla números y el texto `sd`.

In [26]:
pr_num = pd.to_numeric(df_sectores["PR"], errors="coerce")
print("PR no numéricos:", pr_num.isna().sum(), "->", df_sectores.loc[pr_num.isna(), "PR"].unique())

PR no numéricos: 42 -> <StringArray>
['sd']
Length: 1, dtype: str


### 4.4 Duplicados

In [27]:
dup_les = df_lesiones.duplicated().sum()
print(f"df_lesiones – filas idénticas: {dup_les} ({dup_les/len(df_lesiones):.1%})")
print("df_sectores – filas idénticas:", df_sectores.duplicated().sum())
print("df_sectores – coordenadas repetidas:", df_sectores.duplicated(["Latitud", "Longitud"]).sum())
df_sectores[df_sectores.duplicated(["Latitud", "Longitud"], keep=False)].sort_values(["Latitud"])[
    ["ID_MT", "Tramo", "Municipio", "Latitud", "Longitud", "Fallecidos", "GiZScore"]]

df_lesiones – filas idénticas: 294354 (45.7%)
df_sectores – filas idénticas: 0
df_sectores – coordenadas repetidas: 4


,ID_MT,Tramo,Municipio,Latitud,Longitud,Fallecidos,GiZScore
235,4G017,Neiva-Girardot,COYAIMA,"3,728237","-75,078416",15,"3,587306"
315,4G017,Neiva-Girardot,COYAIMA,"3,728237","-75,078416",19,"4,771095"
203,4G026,Chigorodó-Turbo,APARTADÓ,"7,873917","-76,635127",24,"6,250831"
310,4G026,Chigorodó-Turbo,APARTADÓ,"7,873917","-76,635127",15,"3,587306"
202,4G026,Chigorodó-Turbo,TURBO,"8,005361","-76,640697",30,"8,026514"
312,4G026,Chigorodó-Turbo,TURBO,"8,005361","-76,640697",25,"6,546778"
198,4G016,San Pelayo-Lorica,COTORRA,"9,030229","-75,827469",10,"2,10757"
314,4G016,San Pelayo-Lorica,COTORRA,"9,030229","-75,827469",12,"2,699465"


En `df_lesiones` las filas idénticas **no son necesariamente errores**: el dataset no tiene un identificador de víctima ni de evento, por lo que dos lesionados el mismo día en el mismo municipio generan filas iguales. Se documenta como limitación (no se puede distinguir duplicado real de registro legítimo). En `df_sectores` hay 4 pares de puntos con las mismas coordenadas y valores distintos de fallecidos: posibles registros repetidos.

### 4.5 Posibles inconsistencias

**a) Valores de `CANTIDAD` y casos extremos**

In [28]:
print(df_lesiones["CANTIDAD"].describe())
df_lesiones.nlargest(5, "CANTIDAD")

count    644042.000000
mean          1.375421
std           1.626253
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          91.000000
Name: CANTIDAD, dtype: float64


,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,CANTIDAD
422749,25/04/2016,11,BOGOTA D.C.,11001,BOGOTA D.C.,91
380444,02/12/2017,11,BOGOTA D.C.,11001,BOGOTA D.C.,63
421206,16/05/2016,11,BOGOTA D.C.,11001,BOGOTA D.C.,60
378776,29/12/2017,76,VALLE DEL CAUCA,76001,CALI,56
508731,01/01/2011,5,ANTIOQUIA,5001,MEDELLIN,53


**b) Cobertura temporal:** registros y lesionados por año. Se observa si el método de registro cambia en el tiempo.

In [29]:
por_anio = (df_lesiones.assign(ANIO=fecha.dt.year)
            .groupby("ANIO")
            .agg(registros=("CANTIDAD", "size"), lesionados=("CANTIDAD", "sum")))
por_anio["lesionados_por_registro"] = (por_anio["lesionados"] / por_anio["registros"]).round(2)
print("Último mes disponible de", fecha.dt.year.max(), ":", fecha[fecha.dt.year == fecha.dt.year.max()].dt.month.max())
por_anio

Último mes disponible de 2026 : 9


,registros,lesionados,lesionados_por_registro
ANIO,,,
2003,14048,14568,1.04
2004,11822,12257,1.04
2005,10063,10063,1.00
2006,13145,13145,1.00
2007,16194,16194,1.00
2008,17224,17224,1.00
2009,38003,38003,1.00
2010,14744,36004,2.44
2011,14627,36964,2.53


**c) Nombres y códigos de municipio:** nombres repetidos en distintos departamentos.

In [30]:
nombres_multi = df_lesiones.groupby("MUNICIPIO")["COD_MUNI"].nunique()
print("Nombres de municipio asociados a más de un código:", (nombres_multi > 1).sum())
print("Municipios únicos por nombre:", df_lesiones["MUNICIPIO"].nunique(), "| por código:", df_lesiones["COD_MUNI"].nunique())
nombres_multi[nombres_multi > 1].sort_values(ascending=False).head(10)

Nombres de municipio asociados a más de un código: 65


Municipios únicos por nombre: 1003 | por código: 1089


MUNICIPIO
SIN ESTABLECER    6
VILLANUEVA        4
BUENAVISTA        4
LA UNION          4
GUADALUPE         3
CORDOBA           3
GRANADA           3
BOLIVAR           3
ALBANIA           3
SABANALARGA       3
Name: COD_MUNI, dtype: int64

**d) Consistencia entre datasets (tildes):** `df_lesiones` escribe los nombres sin tilde y `df_sectores` con tilde. Un cruce por nombre fallaría.

In [31]:
dep_les = set(df_lesiones["DEPARTAMENTO"].unique())
dep_sec = set(df_sectores["Departamento"].unique())
print("Departamentos de sectores que NO coinciden por nombre con lesiones:", sorted(dep_sec - dep_les))

# Cruce por código DIVIPOLA (ambos leídos como int, por eso coinciden numéricamente)
cod_no_match = set(df_sectores["divipola"]) - set(df_lesiones["COD_MUNI"])
print("Códigos divipola de sectores sin registro en lesiones:", len(cod_no_match), sorted(cod_no_match)[:10])

Departamentos de sectores que NO coinciden por nombre con lesiones: ['ATLÁNTICO', 'BOLÍVAR', 'BOYACÁ', 'CAQUETÁ', 'CÓRDOBA', 'QUINDÍO']
Códigos divipola de sectores sin registro en lesiones: 0 []


**e) Texto: espacios extra y nombres truncados** en `df_sectores`.

In [32]:
print("Tramo truncado (termina en '*'):", df_sectores["Tramo"].str.contains(r"\*", na=False).sum())
print("Nombre con espacios dobles:", df_sectores["Nombre"].str.contains("  ", na=False).sum())
df_sectores.loc[df_sectores["Tramo"].str.contains(r"\*", na=False), "Tramo"].unique()

Tramo truncado (termina en '*'): 15
Nombre con espacios dobles: 25


<StringArray>
['El Hatillo - Girardota - Copacabana - Bello - Med*']
Length: 1, dtype: str

**f) Rangos válidos en `df_sectores`:** coordenadas dentro de Colombia continental/insular (lat ≈ -4,2 a 13,5; lon ≈ -82 a -66) y p-values entre 0 y 1.

In [33]:
fuera = prueba[(~prueba["Latitud"].between(-4.3, 13.5)) | (~prueba["Longitud"].between(-82, -66))]
print("Puntos fuera del rango de Colombia:", len(fuera))
print("p-values fuera de [0,1]:", (~prueba["GiPValue"].between(0, 1)).sum())
print("p-values > 0.05 (no significativos al 95%):", (prueba["GiPValue"] > 0.05).sum())
print("ID_MT = '<Null>' por entidad:"); print(df_sectores[df_sectores["ID_MT"] == "<Null>"]["ENTIDAD"].value_counts())

Puntos fuera del rango de Colombia: 0
p-values fuera de [0,1]: 0
p-values > 0.05 (no significativos al 95%): 60
ID_MT = '<Null>' por entidad:
ENTIDAD
INVIAS-OTROS    126
DEPTO            48
CIUDAD            1
Name: count, dtype: int64


### 4.6 Tabla de hallazgos de calidad

In [34]:
hallazgos = pd.DataFrame([
 ["df_lesiones", "Fecha almacenada como texto", "dtype str en FECHA HECHO; 0 fechas inválidas con formato %d/%m/%Y",
  "No se puede agrupar por año/mes ni ordenar cronológicamente", "Convertir con pd.to_datetime(format='%d/%m/%Y')"],
 ["df_lesiones", "Códigos DIVIPOLA leídos como enteros", "COD_DEPTO/COD_MUNI int64; ceros iniciales perdidos (05001 -> 5001)",
  "Cruces erróneos con otras fuentes DIVIPOLA", "Leer con dtype=str y aplicar zfill(2)/zfill(5)"],
 ["df_lesiones", "Ubicación no establecida", "DEPARTAMENTO='SIN ESTABLECER' (3) y MUNICIPIO='SIN ESTABLECER' (11)",
  "Registros sin asignación geográfica", "Marcar como NaN; excluir de análisis territorial (impacto mínimo)"],
 ["df_lesiones", "Filas idénticas sin identificador", f"{dup_les} filas duplicadas (~45,7%)",
  "Imposible distinguir duplicados reales de víctimas distintas", "No eliminar; documentar y analizar con sum(CANTIDAD)"],
 ["df_lesiones", "Cambio en la forma de registro", "2010-2017: ~2,5 lesionados por registro vs ~1,0 en otros años",
  "Contar filas en lugar de sumar CANTIDAD sesga la tendencia", "Usar siempre la suma de CANTIDAD"],
 ["df_lesiones", "Año 2026 incompleto", "Fecha máxima 30/09/2026",
  "Comparaciones anuales subestiman 2026", "Comparar periodos equivalentes (ene-sep) o excluir 2026"],
 ["df_lesiones", "Nombres de municipio repetidos", "65 nombres con más de un COD_MUNI",
  "Agrupar por nombre mezcla municipios distintos", "Agrupar por COD_MUNI"],
 ["df_lesiones", "Valores extremos en CANTIDAD", "Máximo 91 (Bogotá, 25/04/2016); mediana 1",
  "Distorsionan promedios", "Verificar con la fuente; usar mediana o análisis robusto"],
 ["df_sectores", "Decimales con coma", "GiZScore, GiPValue, Latitud, Longitud como texto",
  "No se pueden calcular estadísticas ni mapear", "read_csv(decimal=',') o str.replace(',', '.') + to_numeric"],
 ["df_sectores", "Faltantes ocultos '<Null>'", "ID_MT='<Null>' en 175 filas (55,4%)",
  "Sectores sin contrato/proyecto asociado", "Reemplazar por NaN (na_values=['<Null>'])"],
 ["df_sectores", "Nulos en Nombre y Tramo", "Nombre: 175 nulos (55,4%); Tramo: 23 nulos (7,3%)",
  "Menor capacidad descriptiva de los sectores", "Imputar 'Sin nombre' o completar desde la fuente"],
 ["df_sectores", "PR con texto 'sd'", "42 valores 'sd' en PR",
  "PR no numérico; no se ubica el punto en la vía", "Reemplazar 'sd' por NaN y convertir a numérico"],
 ["df_sectores", "Coordenadas repetidas", "4 pares de puntos con igual Lat/Lon y Fallecidos distintos",
  "Posible doble conteo de fallecidos", "Revisar y consolidar duplicados"],
 ["df_sectores", "Texto truncado y espacios extra", "15 Tramo terminan en '*'; Nombre con espacios dobles",
  "Agrupaciones por tramo inconsistentes", "str.strip(), normalizar espacios, completar nombres"],
 ["Ambos", "Nombres con/sin tilde", "BOYACÁ vs BOYACA, CÓRDOBA vs CORDOBA, etc.",
  "Un cruce por nombre falla", "Cruzar por código DIVIPOLA o normalizar quitando tildes"],
], columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"])
hallazgos

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,df_lesiones,Fecha almacenada como texto,dtype str en FECHA HECHO; 0 fechas inválidas c...,No se puede agrupar por año/mes ni ordenar cro...,Convertir con pd.to_datetime(format='%d/%m/%Y')
1,df_lesiones,Códigos DIVIPOLA leídos como enteros,COD_DEPTO/COD_MUNI int64; ceros iniciales perd...,Cruces erróneos con otras fuentes DIVIPOLA,Leer con dtype=str y aplicar zfill(2)/zfill(5)
2,df_lesiones,Ubicación no establecida,DEPARTAMENTO='SIN ESTABLECER' (3) y MUNICIPIO=...,Registros sin asignación geográfica,Marcar como NaN; excluir de análisis territori...
3,df_lesiones,Filas idénticas sin identificador,"294354 filas duplicadas (~45,7%)",Imposible distinguir duplicados reales de víct...,No eliminar; documentar y analizar con sum(CAN...
4,df_lesiones,Cambio en la forma de registro,"2010-2017: ~2,5 lesionados por registro vs ~1,...",Contar filas en lugar de sumar CANTIDAD sesga ...,Usar siempre la suma de CANTIDAD
5,df_lesiones,Año 2026 incompleto,Fecha máxima 30/09/2026,Comparaciones anuales subestiman 2026,Comparar periodos equivalentes (ene-sep) o exc...
6,df_lesiones,Nombres de municipio repetidos,65 nombres con más de un COD_MUNI,Agrupar por nombre mezcla municipios distintos,Agrupar por COD_MUNI
7,df_lesiones,Valores extremos en CANTIDAD,"Máximo 91 (Bogotá, 25/04/2016); mediana 1",Distorsionan promedios,Verificar con la fuente; usar mediana o anális...
8,df_sectores,Decimales con coma,"GiZScore, GiPValue, Latitud, Longitud como texto",No se pueden calcular estadísticas ni mapear,"read_csv(decimal=',') o str.replace(',', '.') ..."
9,df_sectores,Faltantes ocultos '<Null>',"ID_MT='<Null>' en 175 filas (55,4%)",Sectores sin contrato/proyecto asociado,Reemplazar por NaN (na_values=['<Null>'])


### 4.7 Limitaciones de los datos frente a la pregunta analítica
- `df_lesiones` solo tiene **lesionados**, no fallecidos ni gravedad, tipo de vehículo, edad o causa del accidente.
- No hay identificador de evento: no se puede contar **accidentes**, solo lesionados.
- El cambio en la forma de registro (2010-2017) y el año 2026 incompleto obligan a cuidar las comparaciones temporales.
- `df_sectores` es una **foto** (sin fecha): no permite ver evolución de los sectores críticos, y solo cubre 22 departamentos y vías nacionales/departamentales.
- Los datasets tienen granularidades distintas (municipio-día vs punto en vía), por lo que solo se pueden relacionar a nivel de **municipio** mediante el código DIVIPOLA.

### 4.8 Vista previa (opcional) de la carga corregida
No es una limpieza completa: solo comprueba que las acciones propuestas de lectura resuelven los problemas de tipo.

In [35]:
df_lesiones_ok = pd.read_csv(DATA_RAW / "lesiones_accidentes_transito.csv",
                             dtype={"COD_DEPTO": str, "COD_MUNI": str},
                             parse_dates=["FECHA HECHO"], date_format="%d/%m/%Y")
df_sectores_ok = pd.read_csv(DATA_RAW / "sectores_criticos_siniestralidad_vial.csv",
                             decimal=",", na_values=["<Null>", "sd"], dtype={"divipola": str})
df_sectores_ok["divipola"] = df_sectores_ok["divipola"].str.zfill(5)
print(df_lesiones_ok.dtypes, "\n")
print(df_sectores_ok.dtypes)

FECHA HECHO     datetime64[us]
COD_DEPTO                  str
DEPARTAMENTO               str
COD_MUNI                   str
MUNICIPIO                  str
CANTIDAD                 int64
dtype: object 

ID_MT               str
ENTIDAD             str
GiZScore        float64
Fallecidos        int64
GiPValue        float64
Tramo               str
Nombre              str
Latitud         float64
Longitud        float64
PR              float64
Municipio           str
Departamento        str
divipola            str
dtype: object


## Conclusión
Ambos datasets son **utilizables** para el proyecto, pero requieren preparación antes del análisis. Los problemas principales son de **tipo de dato y formato** (fechas como texto, códigos DIVIPOLA sin ceros, decimales con coma), de **faltantes ocultos** (`<Null>`, `sd`, `SIN ESTABLECER`) y de **consistencia** entre fuentes (tildes, nombres de municipio repetidos). Ninguno impide continuar; todos se corrigen en la etapa de limpieza. El detalle está en `reports/informe_diagnostico.md`.